# BiasBuster Experiment Pipeline
Baseline + Mitigation + Evaluation

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

RANDOM_STATE = 42


In [ ]:

# Load dataset
df = pd.read_csv("adult.csv")

# Basic cleaning
df = df.replace(" ?", np.nan).dropna()

# Target + sensitive
target = "income"
sensitive = "sex"

# Encode target
df[target] = df[target].apply(lambda x: 1 if ">50K" in x else 0)

df.head()


In [ ]:

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)


In [ ]:

categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", "passthrough", categorical_cols)
    ]
)


In [ ]:

models = {
    "log_reg": LogisticRegression(max_iter=1000),
    "rf": RandomForestClassifier(random_state=RANDOM_STATE),
    "dt": DecisionTreeClassifier(random_state=RANDOM_STATE)
}


In [ ]:

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    results.append([name, acc, prec, rec, f1])

    with open(f"{name}_baseline.pkl", "wb") as f:
        pickle.dump(pipe, f)

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1"])
results_df


In [ ]:

# Reweighting (simple inverse frequency)
group_counts = X_train[sensitive].value_counts()
weights = X_train[sensitive].map(lambda x: 1 / group_counts[x])

reweight_model = LogisticRegression(max_iter=1000)
reweight_pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", reweight_model)
])

reweight_pipe.fit(X_train, y_train, model__sample_weight=weights)

with open("log_reg_reweighted.pkl", "wb") as f:
    pickle.dump(reweight_pipe, f)


In [ ]:

# Save debiased dataset placeholder (same for now)
df.to_csv("processed_dataset.csv", index=False)
